# YOLOv8 Installation

In [ ]:
import ultralytics
ultralytics.checks()

## Download the Model

In [ ]:
# Download YOLOv8 model
!wget https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8m.pt

# Tensorrt

In [ ]:
!pip install tensorrt

In [ ]:
!pip install tensorrt_lean

In [ ]:
!pip install tensorrt_dispatch

In [ ]:
!pip install onnx onnxsim onnxruntime-gpu

In [ ]:
import tensorrt
print(tensorrt.__version__)
assert tensorrt.Builder(tensorrt.Logger())

In [ ]:
pip install numpy==1.26.4

In [ ]:
pip install ultralytics==8.2.38

In [ ]:
# Export YOLOv8 Model to Tensorrt
!yolo export model=yolov8m.pt format=engine half=True device=0

## Inference on Image

In [ ]:
# Inference Using YOLOv8 Model
!yolo detect predict model=yolov8m.pt source="https://ultralytics.com/images/bus.jpg" device=0

In [ ]:
# Inference Using YOLOv8 Tensorrt
!yolo detect predict model=yolov8m.engine source="https://ultralytics.com/images/bus.jpg" device=0

In [ ]:
import cv2
from ultralytics import YOLO

img = cv2.imread("bus.jpg")

#model_path = "yolov8m.pt"
model_path = "yolov8m.engine"

model = YOLO(model_path, task='detect')
classes = model.names
    
outputs = model.predict(img, imgsz=640, conf=0.5, verbose=True, iou=0.7)

detected_conf = outputs[0].boxes.conf.cpu().tolist()
detected_cls = outputs[0].boxes.cls.cpu().int().tolist()
detected_cls = [classes[i] for i in detected_cls]
detected_xyxy = outputs[0].boxes.xyxy.cpu().int().tolist()

---
# ONNX для тритон сервера

In [1]:
pip install onnx==1.15.0

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install ultralytics==8.3.81

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install onnxruntime==1.15.0

  Using cached onnxruntime-1.15.0-cp311-cp311-win_amd64.whl.metadata (4.0 kB)
Using cached onnxruntime-1.15.0-cp311-cp311-win_amd64.whl (6.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
from ultralytics import YOLO

yolo_model = YOLO("yolov8m.pt")

metadata = []
def export_cb(exporter):
    metadata.append(exporter.metadata)

yolo_model.add_callback("on_export_end", export_cb)

#first way of convertion
yolo_model.export(format="onnx",  dynamic=True)

100%|██████████| 49.7M/49.7M [00:00<00:00, 53.2MB/s]


Ultralytics 8.3.81  Python-3.11.8 torch-2.2.1+cu118 CPU (12th Gen Intel Core(TM) i9-12900H)
YOLOv8m summary (fused): 92 layers, 25,886,080 parameters, 0 gradients, 78.9 GFLOPs

PyTorch: starting from 'yolov8m.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (49.7 MB)

ONNX: starting export with onnx 1.15.0 opset 17...
ONNX: slimming with onnxslim 0.1.44...
ONNX: export success  14.3s, saved as 'yolov8m.onnx' (98.9 MB)

Export complete (15.4s)
Results saved to C:\ \ GIT  \TrafficAnalyzer\services\triton
Predict:         yolo predict task=detect model=yolov8m.onnx imgsz=640  
Validate:        yolo val task=detect model=yolov8m.onnx imgsz=640 data=coco.yaml  
Visualize:       https://netron.app


'yolov8m.onnx'

In [3]:
data = """
optimization {
  execution_accelerators {
    gpu_execution_accelerator {
      name: "tensorrt"
      parameters {
        key: "precision_mode"
        value: "FP16"
      }
      parameters {
        key: "max_workspace_size_bytes"
        value: "6221225472"
      }
      parameters {
        key: "trt_engine_cache_enable"
        value: "1"
      }
      parameters {
        key: "trt_engine_cache_path"
        value: "/models/yolo_detector/1"
      }
    }
  }
}
parameters {
  key: "metadata"
  value: {
    string_value: "%s"
  }
}
""" % metadata[0]

with open("config.pbtxt", "w") as f:
    f.write(data)

/models/yolo_detector/1 сюда надо положить файл onnx модели и итоговый config.pbtxt положить в /models/yolo_detector